In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")
secret_value_1 = user_secrets.get_secret("WANDB_API_KEY")

In [ ]:
# 1. Install Requirements
!pip install datasets transformers wandb evaluate scikit-learn -q

import os
import json
import wandb
import numpy as np
import evaluate
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

# 2. Load Secrets
secrets = UserSecretsClient()
os.environ['WANDB_API_KEY'] = secrets.get_secret('WANDB_API_KEY')
login(token=secrets.get_secret('HF_TOKEN'))
wandb.login()

# 3. Load Dataset & id2label mapping
dataset = load_dataset("dair-ai/emotion", "split")
features = dataset['train'].features['label']
id2label = {str(i): name for i, name in enumerate(features.names)}
label2id = {name: i for i, name in enumerate(features.names)}

# 4. Task 3 - Load Tokenizer & Model
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

tokenized_datasets = dataset.map(tokenize_function, batched=True)
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

# 5. Task 4 - Train Version 1 and Version 2
hyperparameters = [
    {"version": "v1", "learning_rate": 3e-5, "epochs": 2},
    {"version": "v2", "learning_rate": 5e-5, "epochs": 2}
]

for hp in hyperparameters:
    print(f"--- Training {hp['version']} ---")
    
    # Initialize W&B run
    wandb.init(
        project='mlops-assignment3',
        name=f"run-{hp['version']}",
        config={'model': model_name, 'platform': 'Kaggle', **hp}
    )
    
    # Reload model for each run to start fresh
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=len(id2label), id2label=id2label, label2id=label2id
    )
    
    training_args = TrainingArguments(
        output_dir=f"./results_{hp['version']}",
        learning_rate=hp['learning_rate'],
        num_train_epochs=hp['epochs'],
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        report_to="wandb",
        run_name=f"run-{hp['version']}"
    )
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_datasets["train"].shuffle(seed=42).select(range(5000)), # Subset for speed
        eval_dataset=tokenized_datasets["validation"],
        compute_metrics=compute_metrics,
    )
    
    trainer.train()
    
    # Task 5: Push best version to Hub (We'll only push v1 for simplicity)
    if hp['version'] == "v1":
        # Replace 'your-username' with your actual HF username
        repo_name = "srajam696/mlops-emotion-distilbert" 
        model.push_to_hub(repo_name)
        tokenizer.push_to_hub(repo_name)
        wandb.run.summary['huggingface_model'] = f"https://huggingface.co/{repo_name}"

    wandb.finish()

HERE I just added the data set whi

In [ ]:
import os
for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        print(os.path.join(root, file))